# 🧹 Manipulación de datos con pandas
**Modelos de Predicción Analítica · Diplomado en Machine Learning y Ciencia de Datos**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leslysandra/primeros-pasos-python/blob/main/notebooks/01_manipulacion_datos_pandas.ipynb)

**Autora:** Ing. Mgr. Lesly Zerna · **Versión:** 2.0 (Python)

> *"Los científicos de datos pasan el 80 % del tiempo limpiando datos. El otro 20 % se quejan de eso."* — chiste clásico de ciencia de datos

En la práctica, los datos crudos casi nunca están listos para modelar. En este notebook aprenderás a **filtrar, seleccionar, ordenar, transformar, resumir y unir tablas** con `pandas`, la librería estándar de la industria para esto en Python.

**Dataset:** `nycflights13` — los 336.776 vuelos que salieron de Nueva York en 2013 (más tablas auxiliares de aerolíneas y aeropuertos).

**Secciones**
1. Encadenamiento de métodos
2. Filtrar filas
3. Seleccionar columnas
4. Ordenar filas
5. Crear y transformar columnas
6. Resumir y agrupar (`groupby`)
7. Unir tablas (`merge`)
8. Cambiar la forma de la tabla (`pivot` / `melt`)
9. Manipular texto
10. Un pipeline completo

> 💡 Las celdas **✏️ Tu turno** tienen espacios `____` para completar. Si aún no completas una, mostrará un error: es normal.

## Preparación

Importamos las librerías y **leemos los datos desde archivos CSV** (descargados directamente desde el repositorio del curso; son ~8 MB, así que la primera carga puede tardar unos segundos).

> Los datos provienen del paquete público `nycflights13`: la tabla `flights` (vuelos) y dos tablas auxiliares, `airlines` (aerolíneas) y `airports` (aeropuertos).

In [ ]:
import numpy as np
import pandas as pd

URL_DATOS = "https://raw.githubusercontent.com/leslysandra/primeros-pasos-python/main/data/"
flights = pd.read_csv(URL_DATOS + "flights.csv.gz")   # pandas descomprime el .gz automáticamente
airlines = pd.read_csv(URL_DATOS + "airlines.csv")
airports = pd.read_csv(URL_DATOS + "airports.csv")

# Función auxiliar para los ejercicios "✏️ Tu turno"
def revisar(condicion, ok="✅ ¡Correcto!", error="❌ Todavía no: revisa tu código e inténtalo otra vez."):
    """Imprime un mensaje según si tu respuesta es correcta."""
    print(ok if condicion else error)

### Un primer vistazo al dataset

Siempre empieza aquí: ¿cuántas filas y columnas hay, y de qué tipo es cada una?

> Trabajaremos con una copia llamada `vuelos` para no modificar el original.

In [ ]:
vuelos = flights.copy()

print(vuelos.shape)   # (filas, columnas): 336.776 filas × 19 columnas
vuelos.info()         # tipos de datos y cantidad de valores no nulos

In [ ]:
vuelos.head()

## Sección 1 — Encadenamiento de métodos (*method chaining*)

En pandas casi todo se hace con **métodos** que se escriben con un punto: `objeto.metodo()`. Como cada método devuelve una tabla (o serie) nueva, puedes **encadenarlos** uno tras otro, de izquierda a derecha, como los pasos de una receta.

Sin encadenar (se lee "de adentro hacia afuera", difícil):

In [ ]:
np.mean(np.sqrt(np.abs(np.array([-4, 9, 16, -25]))))

Encadenando (se lee como una receta):

In [ ]:
pd.Series([-4, 9, 16, -25]).abs().pow(0.5).mean()

Cuando la cadena es larga, la escribimos **entre paréntesis y un método por línea**. Es el estilo que usaremos en todo el curso:

```python
resultado = (
    tabla
    .paso_1(...)
    .paso_2(...)
    .paso_3(...)
)
```

## Sección 2 — Filtrar filas: quedarse con lo que cumple una condición

Filtrar es como el botón "Filtro" de Excel, pero escrito como código. La condición produce una columna de `True`/`False`, y pandas se queda con las filas `True`.

In [ ]:
# Vuelos de enero:
vuelos_enero = vuelos[vuelos["month"] == 1]
print(len(vuelos_enero))   # 27.004

Con **varias condiciones** se usan `&` (y), `|` (o), y **cada condición entre paréntesis**:

In [ ]:
# Vuelos de enero Y desde JFK:
enero_jfk = vuelos[(vuelos["month"] == 1) & (vuelos["origin"] == "JFK")]
print(len(enero_jfk))

# Vuelos con retraso de salida de MÁS de 2 horas:
muy_tarde = vuelos[vuelos["dep_delay"] > 120]
print(len(muy_tarde))

In [ ]:
# Vuelos a Miami O a Dallas → .isin() revisa si el valor está en una lista
mia_dal = vuelos[vuelos["dest"].isin(["MIA", "DAL"])]
print(len(mia_dal))

# Vuelos que NO fueron cancelados (dep_time NO es faltante):
completados = vuelos[vuelos["dep_time"].notna()]
print(len(completados))

Otra forma, a veces más legible, es `.query()`, donde la condición se escribe como texto:

In [ ]:
vuelos.query("month == 1 and origin == 'JFK'").shape

**Operadores de comparación**

| Operador | Significado | | Operador | Significado |
|---|---|---|---|---|
| `==` | igual a | | `!=` | distinto de |
| `>` | mayor que | | `>=` | mayor o igual |
| `<` | menor que | | `<=` | menor o igual |
| `.isin([...])` | está en la lista | | `.isna()` / `.notna()` | es / no es faltante |
| `&` | Y (ambas ciertas) | | `\|` | O (al menos una cierta) |
| `~` | negación (NO) | | | |

### ✏️ Tu turno — Filtrar

In [ ]:
# TODO: ¿cuántos vuelos de FEBRERO salieron de LGA con retraso de salida MAYOR a 30 minutos?
feb_lga = vuelos[(vuelos["month"] == ____) & (vuelos["origin"] == "____") & (vuelos["dep_delay"] > ____)]

print(len(feb_lga))
revisar(len(feb_lga) == 717)

## Sección 3 — Seleccionar columnas: quedarse con (o quitar) variables

Para elegir columnas se usa una **lista de nombres dentro de corchetes dobles** `[[ ]]`.

In [ ]:
# Quedarnos solo con aerolínea, vuelo, origen, destino y retrasos:
solo_retrasos = vuelos[["carrier", "flight", "origin", "dest", "dep_delay", "arr_delay"]]
solo_retrasos.head()

In [ ]:
# Quitar columnas que no necesitamos:
sin_horas = vuelos.drop(columns=["year", "hour", "minute", "time_hour"])
print(sin_horas.shape[1])   # columnas restantes

In [ ]:
# Un rango de columnas adyacentes (de "year" a "day"), con .loc[filas, columnas]:
vuelos.loc[:, "year":"day"].head()

Seleccionar columnas **por patrón en el nombre** con `.filter()`:

In [ ]:
print(vuelos.filter(regex="^dep").columns.tolist())     # empiezan con "dep"
print(vuelos.filter(regex="delay$").columns.tolist())   # terminan en "delay"
print(vuelos.filter(like="arr").columns.tolist())       # contienen "arr"

Renombrar columnas:

In [ ]:
# Con .rename(columns={"nombre_viejo": "nombre_nuevo"}); mantiene todas las demás columnas
vuelos.rename(columns={"dep_delay": "retraso_salida", "arr_delay": "retraso_llegada"}).head(3)

### ✏️ Tu turno — Seleccionar

In [ ]:
# TODO: selecciona todas las columnas cuyo nombre CONTENGA la palabra "time"
columnas_time = vuelos.filter(like="____")

print(columnas_time.columns.tolist())
revisar(columnas_time.shape[1] == 6)

## Sección 4 — Ordenar filas con `sort_values()`

In [ ]:
# De menor a mayor retraso de salida (ascendente es lo normal):
(
    vuelos[["carrier", "flight", "dep_delay"]]
    .sort_values("dep_delay")
    .head(10)
)

In [ ]:
# De mayor a menor (descendente): los peores retrasos registrados
(
    vuelos[["carrier", "flight", "dep_delay"]]
    .sort_values("dep_delay", ascending=False)
    .head(10)
)

In [ ]:
# Por varias columnas: mes y día ascendente, y dentro de cada día el mayor retraso primero
(
    vuelos[["month", "day", "dep_delay"]]
    .sort_values(["month", "day", "dep_delay"], ascending=[True, True, False])
    .head(10)
)

> **Nota:** los valores faltantes (`NaN`) siempre quedan al **final**, sin importar si ordenas de forma ascendente o descendente.

### ✏️ Tu turno — Ordenar

In [ ]:
# TODO: obtén los 5 vuelos con MAYOR tiempo en el aire (air_time), de mayor a menor
top5_air = vuelos.sort_values("____", ascending=____).head(5)

print(top5_air[["carrier", "flight", "origin", "dest", "air_time"]])
revisar(top5_air["air_time"].tolist() == [695.0, 691.0, 686.0, 686.0, 683.0])

## Sección 5 — Crear o transformar columnas

Con `.assign()` agregas columnas nuevas (o sobrescribes existentes). Cada columna nueva se calcula **fila por fila** a partir de las existentes.

In [ ]:
# Convertir el tiempo en el aire de minutos a horas:
(
    vuelos[["air_time", "dest"]]
    .assign(horas_aire=vuelos["air_time"] / 60)
    .head()
)

In [ ]:
# Retraso total (salida + llegada):
(
    vuelos[["dep_delay", "arr_delay"]]
    .assign(retraso_total=vuelos["dep_delay"] + vuelos["arr_delay"])
    .head()
)

Una columna **categórica** a partir de una condición, con `np.where()` (el `SI()` de Excel):

In [ ]:
vuelos_cat = vuelos.assign(
    estado_salida=np.where(vuelos["dep_delay"] > 0, "Con retraso", "A tiempo")
)

vuelos_cat["estado_salida"].value_counts()   # tabla de frecuencias

> ⚠️ **Trampa:** los vuelos cancelados tienen `dep_delay` faltante (`NaN`). Y en Python, `NaN > 0` da `False`, así que esos vuelos quedan clasificados como "A tiempo". Más adelante veremos cómo evitarlo.

Para **varias categorías**, `np.select()` (equivalente a `case_when`): recibe una lista de condiciones, una lista de resultados, y un valor por defecto.

In [ ]:
vuelos_enriquecido = vuelos.assign(
    horas_aire=vuelos["air_time"] / 60,
    retraso_total=vuelos["dep_delay"] + vuelos["arr_delay"],
)

# Las columnas creadas arriba ya existen, así que ahora podemos usarlas.
# (En una MISMA llamada a .assign(), para usar una columna recién creada hay que usar lambda.)
vuelos_enriquecido = vuelos_enriquecido.assign(
    velocidad_mph=lambda d: d["distance"] / d["horas_aire"],
    categoria_retraso=lambda d: np.select(
        [d["dep_delay"] <= 0, d["dep_delay"] <= 30, d["dep_delay"] <= 60],
        ["A tiempo / adelantado", "Retraso menor", "Retraso moderado"],
        default="Retraso grande",
    ),
)

vuelos_enriquecido[["carrier", "flight", "dep_delay", "categoria_retraso", "velocidad_mph"]].head(10)

Si solo quieres **quedarte con las columnas nuevas**, asigna y luego selecciona:

In [ ]:
(
    vuelos
    .assign(horas_aire=vuelos["air_time"] / 60)
    .assign(velocidad_mph=lambda d: d["distance"] / d["horas_aire"])
    [["horas_aire", "velocidad_mph"]]
    .head()
)

### ✏️ Tu turno — Crear columnas

In [ ]:
# TODO: crea la columna `retraso_total` = dep_delay + arr_delay
# y cuenta cuántos vuelos tuvieron un retraso total MAYOR a 60 minutos
vuelos_tot = vuelos.assign(retraso_total=vuelos["dep_delay"] ____ vuelos["arr_delay"])

n_grandes = (vuelos_tot["retraso_total"] > ____).sum()
print(n_grandes)
revisar(n_grandes == 48098)

## Sección 6 — Resumir y agrupar: `groupby()` + `agg()`

- **Resumir** colapsa muchas filas en **una** fila de estadísticas.
- **`groupby()`** divide los datos en grupos para que el resumen se calcule **por grupo**.

Esta combinación es el caballito de batalla del análisis exploratorio.

Primero, un resumen general. Fíjate que pandas **ignora los valores faltantes automáticamente** al calcular `.mean()`, `.median()`, etc.

In [ ]:
resumen_general = pd.DataFrame({
    "total_vuelos": [len(vuelos)],
    "retraso_medio": [vuelos["dep_delay"].mean()],
    "retraso_mediana": [vuelos["dep_delay"].median()],
    "retraso_maximo": [vuelos["dep_delay"].max()],
    # OJO: primero quitamos los NaN con .dropna(); si no, NaN > 0 contaría como "no retrasado"
    "pct_retrasados": [(vuelos["dep_delay"].dropna() > 0).mean() * 100],
})
resumen_general.round(2)

Ahora **por grupo**. Con *named aggregation* nombras cada resultado así: `nombre_nuevo=("columna", "función")`.

In [ ]:
por_aerolinea = (
    vuelos
    .groupby("carrier")
    .agg(
        vuelos=("flight", "size"),                       # size cuenta filas
        retraso_medio=("dep_delay", "mean"),
        pct_retrasados=("dep_delay", lambda s: (s.dropna() > 0).mean() * 100),
    )
    .sort_values("retraso_medio", ascending=False)
    .round(2)
)

por_aerolinea

In [ ]:
# Retraso por mes (¿hay estacionalidad?)
por_mes = vuelos.groupby("month").agg(retraso_medio=("dep_delay", "mean")).round(2)
por_mes   # julio y diciembre suelen ser los peores

Agrupar por **varias variables** (aerolínea × mes). Con `as_index=False` los grupos quedan como columnas normales en vez de índice:

In [ ]:
aerolinea_mes = (
    vuelos
    .groupby(["carrier", "month"], as_index=False)
    .agg(retraso_medio=("dep_delay", "mean"), n_vuelos=("flight", "size"))
)

aerolinea_mes.head(15)

`value_counts()` es el atajo para contar cuántas veces aparece cada valor (equivale a `groupby` + contar):

In [ ]:
print(vuelos["carrier"].value_counts())
print(vuelos.value_counts(["origin", "dest"]).head(10))   # combinaciones origen-destino más frecuentes

### ✏️ Tu turno — Agrupar

In [ ]:
# TODO: calcula el retraso de salida MEDIO por aeropuerto de origen (origin)
# y guarda en `peor_origen` el nombre del origen con el mayor retraso medio
por_origen = vuelos.groupby("____")["dep_delay"].____()

peor_origen = por_origen.idxmax()   # idxmax() devuelve la etiqueta del valor máximo
print(por_origen.round(2))
revisar(peor_origen == "EWR")

## Sección 7 — Unir tablas: `merge()`

Unir (*join*) combina dos tablas usando una **columna clave** que tienen en común.

```
┌──────────────┐   ┌──────────────┐
│   Tabla A    │   │   Tabla B    │
│  clave | val │   │  clave | val │
└──────────────┘   └──────────────┘
```

| `how=` | Qué conserva |
|---|---|
| `"left"` | Todas las filas de A + las coincidentes de B **(el más común)** |
| `"right"` | Todas las filas de B + las coincidentes de A |
| `"inner"` | Solo las filas que coinciden en **ambas** |
| `"outer"` | Todas las filas de ambas |
| *anti-join* | Filas de A que **no** tienen pareja en B (útil para detectar vacíos) → se hace con `indicator=True` |

`nycflights13` trae una tabla auxiliar `airlines` con el nombre completo de cada aerolínea:

In [ ]:
airlines.head()   # carrier | name

In [ ]:
# Agregamos el nombre completo de la aerolínea a cada vuelo:
vuelos_nombre = vuelos.merge(airlines, on="carrier", how="left")

# Buena práctica: verifica que el número de filas NO cambió (un merge mal hecho puede duplicar filas)
print(len(vuelos), len(vuelos_nombre))

vuelos_nombre[["carrier", "name", "flight", "dep_delay"]].head()

Cuando la clave tiene **nombres distintos** en cada tabla, usa `left_on` y `right_on`. Aquí traemos el nombre del aeropuerto de destino:

In [ ]:
aeropuertos_destino = airports[["faa", "name", "lat", "lon"]].rename(columns={"name": "nombre_destino"})

vuelos_nombre2 = vuelos_nombre.merge(
    aeropuertos_destino, left_on="dest", right_on="faa", how="left"
)

vuelos_nombre2[["name", "flight", "dest", "nombre_destino", "dep_delay"]].head()

**Anti-join:** ¿hay vuelos cuyo destino NO aparece en la tabla de aeropuertos? El argumento `indicator=True` crea una columna `_merge` que dice de dónde vino cada fila.

In [ ]:
prueba = vuelos.merge(airports[["faa"]], left_on="dest", right_on="faa", how="left", indicator=True)

sin_aeropuerto = prueba[prueba["_merge"] == "left_only"]   # solo en la tabla izquierda

print(len(sin_aeropuerto))
print(sin_aeropuerto["dest"].unique())   # estos códigos no tienen entrada en `airports`

### ✏️ Tu turno — Unir

In [ ]:
# TODO: une `vuelos` con `airlines` por la columna "carrier" (conservando todos los vuelos)
# y averigua el NOMBRE de la aerolínea con más vuelos
con_nombre = vuelos.merge(airlines, on="____", how="____")

aerolinea_top = con_nombre["name"].value_counts().idxmax()
print(aerolinea_top)
revisar(aerolinea_top == "United Air Lines Inc." and len(con_nombre) == len(vuelos))

## Sección 8 — Cambiar la forma de la tabla: `pivot` y `melt`

**Datos ordenados (*tidy*)**, principio de Hadley Wickham:
- Cada **variable** tiene su propia **columna**.
- Cada **observación** tiene su propia **fila**.
- Cada **valor** tiene su propia **celda**.

| Formato | Qué es | Función |
|---|---|---|
| **Ancho** (*wide*) | Muchas columnas de valores (una por categoría) | `pivot()` : largo → ancho |
| **Largo** (*long*) | Una columna con los nombres y otra con los valores | `melt()` : ancho → largo |

Muchos gráficos y modelos prefieren el formato **largo** o el **ordenado**; para *leer* una tabla, el formato ancho suele ser más cómodo.

Creemos una tabla ancha: retraso medio por aerolínea, con una columna por trimestre.

In [ ]:
retraso_ancho = (
    vuelos
    .assign(trimestre="T" + ((vuelos["month"] + 2) // 3).astype(str))   # (mes+2)//3 = ceil(mes/3)
    .groupby(["carrier", "trimestre"], as_index=False)
    .agg(retraso_medio=("dep_delay", "mean"))
    .round({"retraso_medio": 1})
    .pivot(index="carrier", columns="trimestre", values="retraso_medio")
    .reset_index()
    .rename_axis(columns=None)
)

retraso_ancho   # una fila por aerolínea, una columna por trimestre

`melt()` la devuelve a formato largo (ideal para graficar):

In [ ]:
retraso_largo = retraso_ancho.melt(
    id_vars="carrier",              # columnas que se quedan igual
    var_name="trimestre",           # nueva columna: los nombres de columna antiguos
    value_name="retraso_medio",     # nueva columna: los valores
)

retraso_largo.head()

**Separar y unir columnas de texto.** Por ejemplo, partir `"2013-01-15"` en año, mes y día:

In [ ]:
fechas = pd.DataFrame({"fecha_str": ["2013-01-15", "2013-07-04", "2013-12-25"]})

fechas[["anio", "mes", "dia"]] = fechas["fecha_str"].str.split("-", expand=True)
fechas

In [ ]:
# Lo contrario: unir año, mes y día en una sola columna de TEXTO...
pequeno = vuelos[["year", "month", "day"]].head()
print(pequeno.astype(str).agg("-".join, axis=1))

# ...o, mejor aún, en una columna de FECHAS de verdad (permite ordenar y restar fechas)
print(pd.to_datetime(pequeno))

## Sección 9 — Manipular texto con `.str`

Las columnas de texto tienen un "espacio de nombres" `.str` con funciones para limpiar y transformar cadenas.

In [ ]:
nombres = airlines["name"]
nombres.head()

In [ ]:
print(nombres.str.len().head())           # número de caracteres
print(nombres.str.upper().head(3))        # TODO EN MAYÚSCULAS
print(nombres.str.lower().head(3))        # todo en minúsculas
print(nombres.str.title().head(3))        # Formato Título

In [ ]:
print(nombres.str.contains("Air").head())               # True donde aparece el patrón
print(nombres[nombres.str.contains("Air")].tolist())     # filtrar: nombres que contienen "Air"

In [ ]:
print(nombres.str.replace("Airlines", "AL").head(3))    # reemplaza el texto indicado
print(nombres.str.replace(" ", "_").head(3))            # reemplaza TODAS las apariciones
print(nombres.str.startswith("United").sum())           # ¿cuántos empiezan con "United"?
print(nombres.str.endswith("Inc.").sum())               # ¿cuántos terminan en "Inc."?
print(nombres.str[:6].head(3))                          # primeros 6 caracteres
print(nombres.str.count("a").head(3))                   # cuántas "a" hay en cada nombre
print(pd.Series(["42"]).str.zfill(6)[0])                # rellena con ceros: "000042"

Muy útil para **limpiar datos importados desordenados**:

In [ ]:
desordenado = pd.Series(["  United  ", "delta", "SOUTHWEST", "American   Airlines "])

limpio = (
    desordenado
    .str.strip()                                # quita espacios al inicio y final
    .str.replace(r"\s+", " ", regex=True)       # colapsa espacios repetidos en uno solo
    .str.title()
)
limpio

## Sección 10 — Un pipeline completo

El análisis real encadena muchos pasos. **Objetivo:** producir una tabla limpia y lista para modelar o graficar con el desempeño de cada aerolínea por trimestre.

In [ ]:
desempeno_aerolineas = (
    vuelos

    # 1. Quitar vuelos cancelados (sin hora de salida o de llegada)
    .dropna(subset=["dep_time", "arr_time"])

    # 2. Agregar columnas calculadas útiles
    .assign(
        horas_aire=lambda d: d["air_time"] / 60,
        velocidad_mph=lambda d: d["distance"] / d["horas_aire"],
        retraso_total=lambda d: d["dep_delay"] + d["arr_delay"],
        es_retrasado=lambda d: d["dep_delay"] > 15,       # definición de la FAA de "retrasado"
        trimestre=lambda d: "T" + ((d["month"] + 2) // 3).astype(str),
    )

    # 3. Unir para tener el nombre completo de la aerolínea
    .merge(airlines, on="carrier", how="left")
    .rename(columns={"name": "aerolinea"})

    # 4. Agrupar por aerolínea y trimestre y calcular estadísticas
    .groupby(["aerolinea", "trimestre"], as_index=False)
    .agg(
        n_vuelos=("flight", "size"),
        pct_retrasados=("es_retrasado", lambda s: s.mean() * 100),
        retraso_salida_medio=("dep_delay", "mean"),
        retraso_llegada_medio=("arr_delay", "mean"),
        velocidad_media_mph=("velocidad_mph", "mean"),
    )
    .round(1)

    # 5. Ordenar de peor a mejor puntualidad
    .sort_values("pct_retrasados", ascending=False)
)

desempeno_aerolineas.head(15)

In [ ]:
# Para guardar el resultado como archivo CSV (en Colab queda en la carpeta de archivos de la izquierda):
# desempeno_aerolineas.to_csv("desempeno_aerolineas.csv", index=False)

### ✏️ Tu turno — Mini pipeline

In [ ]:
# TODO: construye un pipeline que:
#   1) se quede con los vuelos de la aerolínea "UA" (United)
#   2) calcule el retraso de salida medio POR ORIGEN
#   3) ordene de mayor a menor
ua_por_origen = (
    vuelos
    [vuelos["carrier"] == "____"]
    .groupby("origin", as_index=False)
    .agg(retraso_medio=("dep_delay", "____"))
    .sort_values("retraso_medio", ascending=____)
)

print(ua_por_origen)
revisar(ua_por_origen.shape == (3, 2) and ua_por_origen["retraso_medio"].is_monotonic_decreasing)

## Referencia rápida: del lenguaje de "verbos" a pandas

| Tarea | Código en pandas |
|---|---|
| Filtrar filas | `df[df["col"] > 10]` · `df.query("col > 10")` |
| Seleccionar columnas | `df[["a", "b"]]` · `df.drop(columns=[...])` · `df.filter(like=...)` |
| Renombrar | `df.rename(columns={"viejo": "nuevo"})` |
| Ordenar | `df.sort_values("col", ascending=False)` |
| Crear columnas | `df.assign(nueva=...)` · `np.where` · `np.select` |
| Resumir por grupo | `df.groupby("g").agg(nombre=("col", "mean"))` |
| Contar | `df["col"].value_counts()` · `groupby(...).size()` |
| Unir tablas | `df1.merge(df2, on="clave", how="left")` |
| Ancho → largo | `df.melt(id_vars=...)` |
| Largo → ancho | `df.pivot(index=..., columns=..., values=...)` |
| Texto | `df["col"].str.lower()` · `.str.contains()` · `.str.replace()` |
| Faltantes | `.isna()` · `.notna()` · `.dropna()` · `.fillna()` |

---

### 🎉 Fin del notebook de manipulación de datos
**Siguiente:** `02_estadistica_descriptiva.ipynb`